In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, stream
from pyspark.sql import functions as F

# t32_stream (5.14): the payment-event stream after at least two producer runs and two stream runs. Read-only.
# Bronze keeps every message; Silver holds valid events once (no duplicate event_id, no broken message, no late event);
# Gold counts failed payments per customer per window exactly as Silver does, and the planted failure burst is an alert.
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat, scfg = cfg["catalog"], stream.load()
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if not ok and detail != "" else ""))

t = lambda k: spark.table(f"{cat}.{stream.TABLES[k]}")
br, sv, gd = t("bronze"), t("silver"), t("gold")
nb, ns, ng = br.count(), sv.count(), gd.count()
print(f"INFO bronze {nb} messages, silver {ns} events, gold {ng} windows with failures")
check("bronze has messages", nb > 0, nb)
check("bronze: one row per Kafka offset", nb == br.select("partition", "offset").distinct().count())
parsed = stream.parse(br)
broken = parsed.filter("event_id IS NULL").count()
check("broken messages kept in bronze, never in silver", broken > 0 and sv.filter("event_id IS NULL").count() == 0, broken)
dup = parsed.filter("event_id IS NOT NULL").groupBy("event_id").count().filter("count > 1").count()
check("silver: event_id unique", ns == sv.select("event_id").distinct().count())
check("duplicates sent to Kafka were removed", dup > 0 and sv.join(
      parsed.groupBy("event_id").count().filter("count > 1").select("event_id"), "event_id").groupBy("event_id").count()
      .filter("count > 1").count() == 0, dup)
late = parsed.join(sv.select("event_id"), "event_id", "left_anti").filter("event_id IS NOT NULL").count()
check("late events (beyond the watermark) were dropped", late > 0, late)
check("silver: amounts in rupees, never negative", sv.filter("amount < 0").count() == 0)
check("no personal columns", not ({"full_name", "phone", "aadhaar", "dob"} & (set(sv.columns) | set(gd.columns))))
check("gold has closed windows", ng > 0, ng)
w = F.window("event_ts", scfg["window"])
want = (sv.filter("status = 'failed'").groupBy(w.alias("w"), "customer_ref").agg(F.count("*").alias("n"))
          .select(F.col("w.start").alias("window_start"), "customer_ref", "n"))
cmp_ = gd.join(want, ["window_start", "customer_ref"], "left").filter("n IS NULL OR n <> failed_payments").count()
check("gold failed_payments = silver failures in the same window", cmp_ == 0, cmp_)
check("gold: one row per window and customer", ng == gd.select("window_start", "customer_ref").distinct().count())
alerts = gd.filter("is_alert").count()
check(f"velocity alert raised (>= {scfg['alert_failed']} failures in {scfg['window']})", alerts > 0, alerts)
check("is_alert matches the threshold", gd.filter(f"is_alert <> (failed_payments >= {scfg['alert_failed']})").count() == 0)
print(f"INFO {alerts} alert windows")
print(f"\nt32_stream: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t32_stream failed")